# Lab 03 — Geometric Transformations (Tasks 1 – 10)
One code cell per task. Run **Setup** first. Every task has a `PATH` you can point at
the real image; when it is missing a synthetic "problem image" is generated so the fix
can be demonstrated. Matrices are built **by hand** (NumPy) as the tasks require;
OpenCV is only used to apply them (`warpAffine` / `warpPerspective`).

Conventions: points are (x, y); arrays are indexed [y, x]; `dsize` = (W, H);
the right-most matrix in a product is applied first; with y pointing down,
the textbook rotation matrix R(+θ) turns the image **clockwise** on screen.

In [ ]:
# ===================== SETUP (run first) =====================
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

os.makedirs("output", exist_ok=True)
np.set_printoptions(precision=3, suppress=True)


def show(imgs, titles, cols=3):
    rows = (len(imgs) + cols - 1) // cols
    plt.figure(figsize=(5 * cols, 4.2 * rows))
    for i, (im, t) in enumerate(zip(imgs, titles)):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(im if im.ndim == 2 else cv2.cvtColor(im, cv2.COLOR_BGR2RGB), cmap="gray")
        plt.title(t, fontsize=10); plt.axis("off")
    plt.tight_layout(); plt.show()


# ---- 3x3 homogeneous builders ----
def T(tx, ty):        return np.array([[1, 0, tx], [0, 1, ty], [0, 0, 1]], float)
def S(sx, sy):        return np.array([[sx, 0, 0], [0, sy, 0], [0, 0, 1]], float)
def Sh(shx=0, shy=0): return np.array([[1, shx, 0], [shy, 1, 0], [0, 0, 1]], float)
def R(deg):
    t = np.deg2rad(deg); c, s = np.cos(t), np.sin(t)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]], float)
def lift(A2, t=(0, 0)):
    """2x2 linear matrix (+ optional translation) -> 3x3 homogeneous."""
    M = np.eye(3); M[:2, :2] = A2; M[:2, 2] = t; return M
def warp(img, M3, size=None, border=0):
    h, w = img.shape[:2]
    return cv2.warpAffine(img, M3[:2].astype(np.float32), size or (w, h),
                          borderValue=(border,) * 3 if img.ndim == 3 else border)


def load(path, maker):
    img = cv2.imread(path) if path else None
    if img is None:
        print(f"'{path}' not found -> synthetic problem image")
        img = maker()
    return img


def textured(h, w, seed=0, label=None):
    """Busy synthetic picture with shapes + text (good for seeing distortions)."""
    rng = np.random.default_rng(seed)
    img = np.full((h, w, 3), 235, np.uint8)
    for x in range(0, w, 40): cv2.line(img, (x, 0), (x, h - 1), (200, 200, 200), 1)
    for y in range(0, h, 40): cv2.line(img, (0, y), (w - 1, y), (200, 200, 200), 1)
    for _ in range(10):
        c = tuple(int(v) for v in rng.integers(0, 200, 3))
        cv2.rectangle(img, tuple(int(v) for v in rng.integers(0, [w, h])),
                      tuple(int(v) for v in rng.integers(0, [w, h])), c, -1)
    if label:
        cv2.putText(img, label, (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.4, (0, 0, 0), 3)
    return img


print("Setup done | OpenCV", cv2.__version__)

In [ ]:
# ===================== TASK 1 — The Tiny Fingerprint (Linear: Scale) =====================
# Build a 2x2 scaling matrix by hand, enlarge 300% in x and y, keep the print centred.
def make_fingerprint():
    img = np.full((120, 120, 3), 255, np.uint8)
    for r in range(4, 55, 5):
        cv2.ellipse(img, (60, 62), (r, int(r * 1.25)), 0, 200, 520, (40, 40, 40), 1)
    return img

img = load("fingerprint.png", make_fingerprint)
h, w = img.shape[:2]
sx = sy = 3.0
S2 = np.array([[sx, 0], [0, sy]])                    # 2x2 LINEAR scale matrix
print("2x2 scale matrix:\n", S2)

# A 2x2 matrix fixes the origin (0,0), so the print grows toward the bottom-right.
# Centring: the old centre c must land on the new centre c', so t = c' - S2 @ c.
c_old = np.array([w / 2, h / 2])

# (a) bigger canvas (3W x 3H) — whole print visible, centred
W3, H3 = int(w * sx), int(h * sy)
t_big = np.array([W3 / 2, H3 / 2]) - S2 @ c_old
big = warp(img, lift(S2, t_big), (W3, H3), 255)

# (b) same canvas (zoom into the centre) — centre stays at the screen centre
t_same = c_old - S2 @ c_old
zoom = warp(img, lift(S2, t_same), (w, h), 255)

no_shift = warp(img, lift(S2), (w, h), 255)            # no translation: grows away from (0,0)
print("translation for 3x canvas:", t_big, "| for same canvas:", t_same)
show([img, no_shift, big, zoom],
     [f"Original {w}x{h}", "2x2 only, same canvas (origin fixed -> off-centre)",
      f"Scaled 300% + centred ({W3}x{H3})", "300% zoom, same canvas"], cols=4)
cv2.imwrite("output/task1_fingerprint_x3.png", big)

In [ ]:
# ===================== TASK 2 — The Dizzy Satellite (Linear: Rotation) =====================
# 2x2 rotation from sin/cos, spin the 45°-rotated image back to north, and compute the
# new canvas size by hand so no corner is chopped.
def make_tilted_city():
    city = textured(300, 400, seed=2, label="N ^ CITY")
    M = cv2.getRotationMatrix2D((200, 150), 45, 1.0)    # simulate the bump: 45° CCW
    t = np.deg2rad(45); nw = int(400 * abs(np.cos(t)) + 300 * abs(np.sin(t)))
    nh = int(400 * abs(np.sin(t)) + 300 * abs(np.cos(t)))
    M[0, 2] += nw / 2 - 200; M[1, 2] += nh / 2 - 150
    return cv2.warpAffine(city, M, (nw, nh), borderValue=(0, 0, 0))

img = load("satellite.png", make_tilted_city)
h, w = img.shape[:2]
theta = 45                                              # image is 45° CCW -> rotate 45° CW back
t = np.deg2rad(theta)
R2 = np.array([[np.cos(t), -np.sin(t)],
               [np.sin(t),  np.cos(t)]])                 # textbook matrix = CW on screen (y down)
print("2x2 rotation matrix:\n", R2)

# New canvas: new_w = w|cos| + h|sin|, new_h = w|sin| + h|cos|
new_w = int(np.ceil(w * abs(np.cos(t)) + h * abs(np.sin(t))))
new_h = int(np.ceil(w * abs(np.sin(t)) + h * abs(np.cos(t))))
print(f"old size {w}x{h} -> new size {new_w}x{new_h}")

c_old, c_new = np.array([w / 2, h / 2]), np.array([new_w / 2, new_h / 2])
t_vec = c_new - R2 @ c_old                             # rotate about the centre, recentre
fixed = warp(img, lift(R2, t_vec), (new_w, new_h))
cropped = warp(img, lift(R2, c_old - R2 @ c_old), (w, h))  # same canvas -> corners cut
show([img, cropped, fixed], ["Tilted satellite (45°)", "Rotated back, same canvas (cut)",
                             f"Rotated back, {new_w}x{new_h} canvas"])
cv2.imwrite("output/task2_satellite_north.png", fixed)
# If your real photo is tilted the other way, use theta = -45.

In [ ]:
# ===================== TASK 3 — The Fast Train Barcode (Linear: Shear) =====================
# The barcode leans right (top shifted right). Build a 2x2 horizontal shear to push it back.
def make_slanted_barcode():
    rng = np.random.default_rng(4)
    bc = np.full((200, 360, 3), 255, np.uint8)
    x = 30
    while x < 330:
        bw = int(rng.integers(2, 9)); cv2.rectangle(bc, (x, 30), (x + bw, 170), (0, 0, 0), -1)
        x += bw + int(rng.integers(2, 7))
    k = 0.4; h = bc.shape[0]                            # x' = x + k*(h - y): top moves right
    M = np.array([[1, -k, k * h], [0, 1, 0]], np.float32)
    return cv2.warpAffine(bc, M, (int(360 + k * h), h), borderValue=(255, 255, 255))

img = load("barcode.png", make_slanted_barcode)
h, w = img.shape[:2]
k = 0.4                                                 # slant factor (measure: x-shift / height)
Sh2 = np.array([[1, k],
                [0, 1]])                                # x' = x + k*y : pushes lower rows right
# The slant shifted the TOP right by k*(h - y); applying +k*y and moving left by k*h undoes it:
fixed = warp(img, lift(Sh2, (-k * h, 0)), (w, h), 255)
print("2x2 shear matrix:\n", Sh2, "\ntranslation:", (-k * h, 0))
show([img, fixed], ["Slanted barcode", "Shear-corrected (rectangular bars)"], cols=2)
cv2.imwrite("output/task3_barcode_fixed.png", fixed)

In [ ]:
# ===================== TASK 4 — The Hidden Treasure Map (Affine: Translation) =====================
# The 'X' is 150 px to the LEFT and 80 px UP (off-screen). A 2x2 matrix cannot move pixels
# (origin is fixed), so upgrade to 3x3 homogeneous and slide everything +150 right, +80 down.
def make_map():
    m = np.full((400, 600, 3), (170, 220, 240), np.uint8)
    cv2.polylines(m, [np.array([[60, 80], [300, 40], [520, 160], [430, 360], [90, 330]], np.int32)],
                  True, (60, 120, 60), 6)
    cv2.line(m, (90, 60), (130, 100), (0, 0, 255), 6); cv2.line(m, (130, 60), (90, 100), (0, 0, 255), 6)
    cv2.putText(m, "X marks the spot", (40, 140), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 150), 2)
    return m

world = load("map.png", make_map)
Tmis = T(-150, -80)                    # how the map got misaligned (for the synthetic demo)
misaligned = warp(world, Tmis)         # what the user sees: X is off-screen
Tfix = T(150, 80)                      # the fix: +150 right, +80 down
print("Translation matrix:\n", Tfix)
print("Tfix @ Tmis =\n", Tfix @ Tmis, "(identity -> map restored)")
restored = warp(world, Tfix @ Tmis)    # compose on the source data: no pixels lost
naive = warp(misaligned, Tfix)         # translating the already-cropped view loses the X
show([misaligned, naive, restored], ["Misaligned view (X hidden)",
                                     "T applied to cropped view (data lost)",
                                     "T @ misalignment on source map (X visible)"])
cv2.imwrite("output/task4_map_fixed.png", restored)
# With a real misaligned image (content still inside the array), just: warp(img, T(150, 80)).

In [ ]:
# ===================== TASK 5 — The Robot's Assembly Line (Rigid Transformation) =====================
# Rotation + translation combined into ONE 3x3 matrix (shape and size preserved).
def make_chip_scene():
    img = np.full((400, 600, 3), (90, 90, 90), np.uint8)
    chip = np.zeros((80, 140, 3), np.uint8); chip[:] = (30, 30, 30)
    for x in range(10, 140, 16):
        cv2.rectangle(chip, (x, 0), (x + 6, 8), (200, 200, 200), -1)
        cv2.rectangle(chip, (x, 72), (x + 6, 80), (200, 200, 200), -1)
    cv2.putText(chip, "CPU", (40, 52), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 200, 255), 2)
    canvas = np.zeros_like(img); canvas[160:240, 230:370] = chip
    M = T(420, 120) @ R(-60) @ T(-300, -200)           # chip rotated 60° CCW, off-centre
    placed = warp(canvas, M); m = warp((canvas.sum(2) > 0).astype(np.uint8) * 255, M)
    img[m > 0] = placed[m > 0]
    return img

img = load("chip.png", make_chip_scene)
h, w = img.shape[:2]
chip_center, chip_angle = (420, 120), 60    # measured: centre (x,y) and CCW tilt in degrees
gripper = (w / 2, h / 2)                    # gripper waits at the image centre

# rotate about the chip centre by +60° CW (textbook R(+θ)), then move the centre to the gripper
M_rigid = T(*gripper) @ R(chip_angle) @ T(-chip_center[0], -chip_center[1])
print("Single rigid 3x3 matrix:\n", M_rigid)
print("det(2x2) =", round(np.linalg.det(M_rigid[:2, :2]), 6), "(1 -> no scaling, no reflection)")
aligned = warp(img, M_rigid, border=90)
for im in (img, aligned):
    cv2.drawMarker(im, (int(gripper[0]), int(gripper[1])), (0, 255, 0), cv2.MARKER_CROSS, 40, 2)
show([img, aligned], ["Chip rotated & off-centre (green = gripper)", "After one rigid transform"], cols=2)
cv2.imwrite("output/task5_chip_aligned.png", aligned)

In [ ]:
# ===================== TASK 6 — The Architect's Blueprint (Similarity Transformation) =====================
# Scale + rotation + translation in ONE 3x3 matrix (angles preserved, size changes).
def make_blueprint_scene():
    bp = np.full((300, 400, 3), (140, 70, 20), np.uint8)
    cv2.rectangle(bp, (60, 80), (340, 260), (255, 255, 255), 3)
    cv2.line(bp, (60, 80), (200, 20), (255, 255, 255), 3); cv2.line(bp, (200, 20), (340, 80), (255, 255, 255), 3)
    cv2.rectangle(bp, (170, 180), (230, 260), (255, 255, 255), 2)
    cv2.putText(bp, "HOUSE PLAN", (110, 150), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
    M = T(110, 90) @ R(-35) @ S(0.4, 0.4) @ T(-200, -150)   # small, rotated, in a corner
    return warp(bp, M, (400, 300), 255)

img = load("blueprint.png", make_blueprint_scene)
h, w = img.shape[:2]
bp_center, bp_angle, bp_scale = (110, 90), 35, 0.4    # measured: centre, CCW tilt, current scale
s = 1 / bp_scale                                       # grow back to full size
M_sim = T(w / 2, h / 2) @ R(bp_angle) @ S(s, s) @ T(-bp_center[0], -bp_center[1])
print("Single similarity 3x3 matrix:\n", M_sim)
print(f"det(2x2) = {np.linalg.det(M_sim[:2, :2]):.3f} = s^2 = {s ** 2:.3f}")
fixed = warp(img, M_sim, border=255)
show([img, fixed], ["Imported blueprint (small, rotated, corner)", "Fixed in one similarity step"], cols=2)
cv2.imwrite("output/task6_blueprint_fixed.png", fixed)

In [ ]:
# ===================== TASK 7 — The Glitched Image (General Affine from 3 points) =====================
# 3 landmark pairs -> 6 equations -> solve for the 6 affine unknowns, then fix the image.
original = textured(300, 400, seed=7, label="LANDMARKS")
glitch = T(80, 90) @ R(-15) @ Sh(0.35, 0) @ S(0.7, 0.6)     # squish + slant + rotate + move
img = cv2.imread("glitched.png")
if img is None:
    print("'glitched.png' not found -> synthetic glitched image")
    img = warp(original, glitch, (520, 380), border=255)

correct = np.float32([[50, 50], [350, 60], [80, 260]])        # landmarks in the correct photo
pts_h = np.vstack([correct.T, np.ones(3)])
glitched = (glitch @ pts_h)[:2].T.astype(np.float32)          # same landmarks in glitched photo
print("Landmarks (glitched -> correct):\n", np.hstack([glitched, correct]))

# Each pair (x,y)->(u,v):  u = a x + b y + c ,  v = d x + e y + f
A, bvec = [], []
for (x, y), (u, v) in zip(glitched, correct):
    A.append([x, y, 1, 0, 0, 0]); bvec.append(u)
    A.append([0, 0, 0, x, y, 1]); bvec.append(v)
params = np.linalg.solve(np.array(A, float), np.array(bvec, float))
M_aff = np.vstack([params.reshape(2, 3), [0, 0, 1]])
print("Solved affine matrix:\n", M_aff)
print("Matches cv2.getAffineTransform:",
      np.allclose(M_aff[:2], cv2.getAffineTransform(glitched, correct), atol=1e-4))
repaired = warp(img, M_aff, (original.shape[1], original.shape[0]), border=255)
vis = img.copy()
for p in glitched: cv2.circle(vis, tuple(int(v) for v in p), 7, (0, 0, 255), -1)
show([original, vis, repaired], ["Reference", "Glitched (red = landmarks)", "Repaired"])
cv2.imwrite("output/task7_repaired.png", repaired)

In [ ]:
# ===================== TASK 8 — The Sidewalk Illusion (Projective: Perspective) =====================
# 4 corners that should form a square -> perspective matrix -> bird's-eye view.
def make_chalk_photo():
    art = textured(400, 400, seed=8, label="CHALK ART")
    cv2.rectangle(art, (0, 0), (399, 399), (0, 0, 255), 8)
    src = np.float32([[0, 0], [400, 0], [400, 400], [0, 400]])
    dst = np.float32([[230, 120], [470, 120], [640, 430], [60, 430]])  # far edge shorter
    street = np.full((480, 700, 3), (110, 110, 110), np.uint8)
    P = cv2.getPerspectiveTransform(src, dst)
    warped = cv2.warpPerspective(art, P, (700, 480))
    m = cv2.warpPerspective(np.full((400, 400), 255, np.uint8), P, (700, 480))
    street[m > 0] = warped[m > 0]
    return street

img = load("chalk_art.jpg", make_chalk_photo)
corners = np.float32([[230, 120], [470, 120], [640, 430], [60, 430]])   # TL, TR, BR, BL (pick with plt hover)
side = 400
square = np.float32([[0, 0], [side - 1, 0], [side - 1, side - 1], [0, side - 1]])
P = cv2.getPerspectiveTransform(corners, square)                       # 3x3, 8 DOF
print("Perspective matrix:\n", P)
birds_eye = cv2.warpPerspective(img, P, (side, side))
vis = img.copy(); cv2.polylines(vis, [corners.astype(np.int32)], True, (0, 255, 0), 3)
show([vis, birds_eye], ["Photo + chosen corners", "Bird's-eye view"], cols=2)
cv2.imwrite("output/task8_birds_eye.png", birds_eye)

In [ ]:
# ===================== TASK 9 — The Stadium Panorama (Projective: Homography) =====================
# 4 matching points in both photos -> homography -> warp camera 2 into camera 1's space.
def make_stadium_pair():
    field = np.full((400, 900, 3), (40, 140, 40), np.uint8)
    for x in range(0, 900, 90): field[:, x:x + 45] = (50, 160, 50)          # mowing stripes
    cv2.rectangle(field, (20, 20), (880, 380), (255, 255, 255), 4)
    cv2.line(field, (450, 20), (450, 380), (255, 255, 255), 4)
    cv2.circle(field, (450, 200), 70, (255, 255, 255), 4)
    cv2.rectangle(field, (20, 110), (150, 290), (255, 255, 255), 4)
    cv2.rectangle(field, (750, 110), (880, 290), (255, 255, 255), 4)
    left = field[:, :520].copy()
    P2 = np.array([[1.0, 0.05, -380], [0.02, 1.0, 0], [0.0002, 0, 1]])      # cam 2: shifted + tilted
    right = cv2.warpPerspective(field, P2, (520, 400))
    return left, right, P2

left, right = cv2.imread("cam1.jpg"), cv2.imread("cam2.jpg")
if left is None or right is None:
    print("Camera photos not found -> synthetic stadium pair")
    left, right, P2 = make_stadium_pair()
    pts_left = np.float32([[450, 20], [450, 380], [400, 200], [500, 200]])   # in camera 1
    ph = np.vstack([pts_left.T, np.ones(4)]); q = P2 @ ph
    pts_right = (q[:2] / q[2]).T.astype(np.float32)                           # same points, camera 2
else:   # REPLACE the zeros: hover in plt.imshow to read (x, y) — same 4 physical points, same order
    pts_left = np.float32([[0, 0], [0, 0], [0, 0], [0, 0]])
    pts_right = np.float32([[0, 0], [0, 0], [0, 0], [0, 0]])

H = cv2.getPerspectiveTransform(pts_right, pts_left)     # exactly 4 pts (findHomography also works)
print("Homography (camera 2 -> camera 1):\n", H)
pano_w = left.shape[1] + right.shape[1]
pano = cv2.warpPerspective(right, H, (pano_w, left.shape[0]))
pano[:left.shape[0], :left.shape[1]] = left              # paste camera 1 on top
cols = np.where(pano.max(axis=(0, 2)) > 0)[0]
pano = pano[:, :cols.max() + 1]                          # crop empty black columns
vl, vr = left.copy(), right.copy()
for p, q in zip(pts_left, pts_right):
    cv2.circle(vl, tuple(int(v) for v in p), 8, (0, 0, 255), -1)
    cv2.circle(vr, tuple(int(v) for v in q), 8, (0, 0, 255), -1)
show([vl, vr, pano], ["Camera 1 (+4 points)", "Camera 2 (+4 points)", "Stitched panorama"])
cv2.imwrite("output/task9_panorama.png", pano)

In [ ]:
# ===================== TASK 10 — The Master Forger (Full Hierarchy Challenge) =====================
# Linear scale -> rigid move near the frame -> projective warp of the 4 corners into the frame.
def make_wall():
    wall = np.full((500, 700, 3), (180, 200, 215), np.uint8)
    frame = np.int32([[380, 90], [600, 130], [590, 400], [385, 430]])        # angled empty frame
    cv2.fillPoly(wall, [frame], (240, 240, 240))
    cv2.polylines(wall, [frame], True, (30, 60, 110), 14)
    return wall, frame.astype(np.float32)

wall = cv2.imread("museum_wall.jpg")
if wall is None:
    print("Museum wall not found -> synthetic wall with an angled frame")
    wall, frame_pts = make_wall()
else:
    frame_pts = np.float32([[0, 0], [0, 0], [0, 0], [0, 0]])   # REPLACE: TL, TR, BR, BL of the frame
painting = cv2.imread("painting.jpg")
if painting is None:
    painting = textured(600, 480, seed=10, label="MONA?")
ph, pw = painting.shape[:2]
H, W = wall.shape[:2]

# Step 1 — LINEAR scale (2x2): shrink to roughly the frame size
target_h = np.linalg.norm(frame_pts[3] - frame_pts[0])
s = target_h / ph
M_scale = lift(np.array([[s, 0], [0, s]]))

# Step 2 — RIGID: small rotation + translation so the painting sits near the frame
frame_c = frame_pts.mean(axis=0)
M_rigid = T(*frame_c) @ R(-5) @ T(-pw * s / 2, -ph * s / 2)
M_affine_chain = M_rigid @ M_scale                         # scale first, then rigid
stage1 = warp(painting, M_scale, (W, H))
stage2 = warp(painting, M_affine_chain, (W, H))

# Step 3 — PROJECTIVE: where are the painting's corners now? map them onto the frame corners
corners = np.float32([[0, 0], [pw, 0], [pw, ph], [0, ph]])
moved = (M_affine_chain @ np.vstack([corners.T, np.ones(4)]))[:2].T.astype(np.float32)
P = cv2.getPerspectiveTransform(moved, frame_pts)
M_total = P @ M_affine_chain                               # whole hierarchy in ONE 3x3
print("Scale:\n", M_scale, "\nRigid:\n", M_rigid, "\nProjective:\n", P, "\nTotal:\n", M_total)

warped = cv2.warpPerspective(painting, M_total, (W, H))
mask = np.zeros((H, W), np.uint8)
cv2.fillConvexPoly(mask, frame_pts.astype(np.int32), 255)
result = wall.copy(); result[mask == 255] = warped[mask == 255]
cv2.polylines(result, [frame_pts.astype(np.int32)], True, (30, 60, 110), 14)  # redraw frame on top
show([painting, stage1, stage2, result],
     ["Painting", "1. Linear scale", "2. + Rigid (near frame)", "3. + Projective (forged!)"], cols=4)
cv2.imwrite("output/task10_forgery.png", result)